# Chapter 10 · Success metrics

Is the store's agent ready for a prototype, a pilot or production? This notebook uses neither the model nor the GPU: it reads the results measured in chapters 7, 8 and 9, with three runs per variant, and turns them into a metrics contract (outcome, guardrails and counter-metric) and into a decision.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. The assumptions, the stages and the saved results

Replace `ASSUMPTIONS` with your team's and run the notebook again.

In [ ]:
"""Chapter 10: success metrics. Prototype, pilot or production?

It does not use the model: it reads the results already measured in chapters 7, 8 and 9
(data/results) and uses them to decide what stage the store's agent is at.
"""
import io
import json
import math
import sys
import urllib.request
import zipfile
from pathlib import Path

ROOT = Path.cwd()                           # where data/ is
REPOSITORY = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
              "archive/refs/heads/main.zip")

# Cost assumptions: a person's minutes and prices. Replace them with your own.
ASSUMPTIONS = {
    "minutes_by_hand": 60,       # a task done entirely by a person
    "minutes_review": 10,        # reviewing a change the agent says is done
    "minutes_touch_up": 20,      # finishing a good change the agent didn't call done
    "minutes_read_report": 5,    # reading what the agent left before doing it by hand
    "minutes_false_done": 90,    # finding a false "done" later and fixing it
    "dollars_per_person_hour": 30,
    "dollars_per_gpu_hour": 0.40,
}

# What each stage requires. In the pilot a person reviews every change, so the
# measured rate of false "done"s is enough; in production nobody does, and what counts
# is the upper bound of its interval: the worst it can be given what has been seen.
STAGES = {
    "prototype": {"finished_min": 0.20},
    "pilot": {"finished_min": 0.30, "false_max": 0.05, "cost_max": 1.00},
    "production": {"finished_min": 0.50, "false_upper_max": 0.01, "cost_max": 0.70,
                   "range_max": 0.10},
}



def prepare():
    """The saved results. If they are missing (in Colab), they are downloaded from the
    book's repository."""
    if not (ROOT / "data" / "results").exists():
        with urllib.request.urlopen(REPOSITORY) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        for name in archive.namelist():
            relative = name.split("/", 1)[1]
            if relative.startswith("en/data/results/") and not name.endswith("/"):
                destination = ROOT / relative.removeprefix("en/")
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(name))


def results(chapter):
    """The results of a chapter's repetitions, as they were saved."""
    path = ROOT / "data" / "results" / f"chapter_{chapter:02d}.json"
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
prepare()

## 2. Each attempt, in one of four classes

In [ ]:
def classify(row):
    """What a person has to do after one of the agent's attempts:
    - "good": the agent says it is done and it is true; it has to be reviewed;
    - "false": it says it is done and it is not true; someone will find out late;
    - "touch up": the change is right, but the agent didn't call it good (for example,
      the new test was missing); it has to be finished;
    - "by hand": it is not right and the agent doesn't say it is; it has to be done."""
    if row["says_done"]:
        return "good" if row["solved"] else "false"
    return "touch up" if row["solved"] else "by hand"


def person_minutes(cls, s=ASSUMPTIONS):
    """The minutes of a person that an attempt of each class leaves behind."""
    return {"good": s["minutes_review"],
            "false": s["minutes_review"] + s["minutes_false_done"],
            "touch up": s["minutes_read_report"] + s["minutes_touch_up"],
            "by hand": s["minutes_read_report"] + s["minutes_by_hand"]}[cls]


def interval(hits, total, z=1.96):
    """The 95% confidence interval of a proportion (Wilson's): the values the real
    rate can lie between after seeing "hits" out of "total"."""
    if total == 0:
        return 0.0, 1.0
    p = hits / total
    center = (p + z * z / (2 * total)) / (1 + z * z / total)
    margin = (z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total))
              / (1 + z * z / total))
    return max(0.0, center - margin), min(1.0, center + margin)

In [ ]:
print(interval(3, 72))   # three false "done"s in 72 attempts

## 3. The metrics contract and the stage

In [ ]:
def metrics(runs, s=ASSUMPTIONS):
    """An agent's metrics from several runs (a list of lists of rows, one per seed):
    the outcome, the guardrails and the counter-metric."""
    rows = [r for run in runs for r in run]
    n = len(rows)
    classes = [classify(r) for r in rows]
    finished = sum(c in ("good", "touch up") for c in classes)
    false_dones = classes.count("false")
    per_run = [sum(r["solved"] for r in run) / len(run) for run in runs]
    minutes = sum(person_minutes(c, s) for c in classes)
    gpu_hours = sum(r["seconds"] for r in rows) / 3600
    dollars = (minutes / 60 * s["dollars_per_person_hour"]
               + gpu_hours * s["dollars_per_gpu_hour"])
    by_hand = n * s["minutes_by_hand"] / 60 * s["dollars_per_person_hour"]
    return {"attempts": n, "classes": {c: classes.count(c) for c in
                                       ("good", "touch up", "false", "by hand")},
            # outcome
            "finished": finished / n,
            # guardrails
            "false_done": false_dones / n,
            "false_done_interval": interval(false_dones, n),
            "range": max(per_run) - min(per_run),
            # counter-metric
            "minutes_per_task": minutes / n, "gpu_hours": gpu_hours,
            "dollars_per_finished": dollars / max(finished, 1),
            "cost_vs_by_hand": dollars / by_hand}


def stage(m, stages=STAGES):
    """The highest stage whose thresholds the agent meets, and why it goes no higher."""
    reached, reason = "none", None
    for name, threshold in stages.items():
        failures = []
        if m["finished"] < threshold.get("finished_min", 0):
            failures.append(f"it finishes {m['finished']:.0%} and "
                            f"{threshold['finished_min']:.0%} is needed")
        if m["false_done"] > threshold.get("false_max", 1):
            failures.append(f"{m['false_done']:.1%} of attempts are false \"done\"s, "
                            f"above {threshold['false_max']:.0%}")
        if m["false_done_interval"][1] > threshold.get("false_upper_max", 1):
            failures.append(f"false \"done\"s could reach "
                            f"{m['false_done_interval'][1]:.0%}, above "
                            f"{threshold['false_upper_max']:.0%}")
        if m["cost_vs_by_hand"] > threshold.get("cost_max", math.inf):
            failures.append(f"it costs {m['cost_vs_by_hand']:.0%} of doing it by hand")
        if m["range"] > threshold.get("range_max", 1):
            failures.append(f"it varies {m['range']:.0%} between runs")
        if failures:
            reason = (name, failures)
            break
        reached = name
    return reached, reason

## 4. The book's agents

In [ ]:
def chapter_7_runs(variant):
    """The three sampling runs of a chapter 7 variant."""
    data = results(7)
    return [data[f"{variant} | seed {s}"]["rows"] for s in (1, 2, 3)]


AGENTS = {"chapter 6 agent": "no frame",
          "chapter 7 agent": "frame and plan"}


def report(s=ASSUMPTIONS, stages=STAGES):
    """The metrics and the stage of the two agents measured with three runs."""
    rows = {}
    for name, variant in AGENTS.items():
        m = metrics(chapter_7_runs(variant), s)
        rows[name] = dict(m, stage=stage(m, stages))
    return rows


def guardrails_from_other_chapters():
    """Two more guardrails, measured in chapters 8 and 9: whether the repository stays
    green when a batch is split up, and whether the store's decisions are respected."""
    batches = results(8)
    green = {form: sum(batch[form]["visible"] for batch in batches.values())
             for form in ("last wins", "same copy", "in waves", "with integrator")}
    requests = results(9)
    respected = {}
    for key, data in requests.items():
        form = key.split(" | ")[0]
        respected[form] = (respected.get(form, 0)
                           + sum(r["criterion"] for r in data["rows"]))
    return {"batches": len(batches), "green_batches": green,
            "attempts_per_form": len(requests) // 4 * 5,
            "decisions_respected": respected}

In [ ]:
for name, m in report().items():
    print(f"{name}: finishes {m['finished']:.0%}, false {m['false_done']:.1%} "
          f"(between {m['false_done_interval'][0]:.0%} and "
          f"{m['false_done_interval'][1]:.0%}), "
          f"cost {m['cost_vs_by_hand']:.0%} of doing it by hand → {m['stage'][0]}")
    if m["stage"][1]:
        print("   does not reach", m["stage"][1][0] + ":", "; ".join(m["stage"][1][1]))
print(guardrails_from_other_chapters())

## 5. Try it: smaller tasks

In [ ]:
smaller = dict(ASSUMPTIONS, minutes_by_hand=30)
m = report(smaller)["chapter 7 agent"]
print(f"{m['cost_vs_by_hand']:.0%} of doing it by hand → {m['stage'][0]}")

## Exercises

1. For each of the chapter's metrics, say whether it is an outcome, a guardrail or a counter-metric, and give an example of how one could be improved at the expense of another. *(No GPU.)*

2. Replace the assumptions with your team's: how long a small task takes by hand, how much it costs to review a change and how much an error found late. What stage does the chapter 7 agent end up at? *(No GPU.)*

3. Which assumption weighs most on the cost? Vary one at a time, to double and to half, and see how much the cost compared with doing it by hand changes. *(No GPU.)*

4. Use `interval` to work out how many attempts without a single false "done" would be needed for the upper bound to drop below 5% and below 1%. How many weeks of pilot would that be, if the agent does twenty tasks a day? *(No GPU.)*

5. Compute the metrics of the four ways of asking of chapter 9, counting as a false "done" also the attempts that do not respect the store's decisions. Does the stage change? *(No GPU.)*

6. The contract does not look at how long the agent takes on each task. Add a time metric, with the seconds of each row, and a reasonable threshold for production. Does the chapter 7 agent meet it? *(No GPU.)*

7. Make the report of the single greedy run of chapter 7 (the file `chapter_07_greedy.json` in `data/results`) instead of the three with sampling. Does the stage change? What does that tell you about deciding with a single run? *(No GPU.)*

8. Write the exit conditions of the pilot as a function that receives the numbers of a week and says whether to go on, stop, or propose the move to production. *(No GPU.)*

9. Which of the chapter's metrics is hardest to measure in a real pilot, where there are no hidden tests? Propose how to measure it. *(No GPU.)*

10. **Project.** Write the metrics contract for a coding agent you use or would like to use in your team: outcome, guardrails and counter-metric, with the thresholds of each stage, before measuring anything. Measure for a week and decide.

## References

*Trustworthy Online Controlled Experiments* (Kohavi, Tang and Xu, 2020). [Wilson (1927)](https://doi.org/10.1080/01621459.1927.10502953), the confidence interval of a proportion. The North Cycles store, its program and its tasks are fictional. Original code and experiments.